In [1]:
import psycopg2
from pathlib import Path
import pandas as pd
import json
import requests

In [2]:
conn = psycopg2.connect(
    host = "localhost", port="5432"
    , dbname = 'postgres', user = "postgres", password = "1234"
)

In [3]:
try:
    cur  = conn.cursor()
    d = cur.execute("select * from orders;")
    orders =  cur.fetchall()
except Exception as e : 
    print(f"Exception occurred as {e}")
    conn.rollback()
finally:
    cur.close()
    conn.commit() 

In [4]:
orders

[(101, 1, 1, Decimal('1000.00'), 1),
 (102, 2, 2, Decimal('500.00'), 2),
 (103, 3, 3, Decimal('150.00'), 1),
 (104, 4, 5, Decimal('200.00'), 1),
 (105, 3, 3, Decimal('150.00'), 2)]

In [5]:
RAW       = Path("data/raw")
PROCESSED = Path("data/processed")
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)


In [6]:
import sqlite3
DB_FILE = Path("data/source.db")
conn = sqlite3.connect(DB_FILE)
cur  = conn.cursor()

In [7]:
cur.executescript("""
DROP TABLE IF EXISTS transactions;
DROP TABLE IF EXISTS employees;

CREATE TABLE transactions (
    txn_id      INTEGER PRIMARY KEY,
    customer_id INTEGER,
    product     TEXT,
    amount      REAL,
    status      TEXT,
    created_at  TEXT,
    updated_at  TEXT
);

CREATE TABLE employees (
    emp_id     INTEGER PRIMARY KEY,
    name       TEXT,
    department TEXT,
    salary     REAL,
    hire_date  TEXT
);
""")

In [8]:
cur.executemany("INSERT INTO transactions VALUES (?,?,?,?,?,?,?)", [
    (1, 1, "Laptop",     999.99, "completed", "2024-01-10", "2024-01-10 09:00"),
    (2, 2, "Keyboard",    79.99, "completed", "2024-01-15", "2024-01-15 14:00"),
    (3, 3, "Monitor",    349.99, "shipped",   "2024-02-01", "2024-02-02 08:00"),
    (4, 1, "Mouse",       25.00, "completed", "2024-02-10", "2024-02-10 16:00"),
    (5, 4, "Tablet",     599.99, "completed", "2024-02-20", "2024-02-20 11:00"),
    (6, 5, "Headphones", 199.99, "completed", "2024-03-01", "2024-03-01 09:00"),
    (7, 2, "USB Hub",     49.99, "pending",   "2024-03-05", "2024-03-05 13:00"),
    (8, 6, "Workstation",1299.99,"completed", "2024-03-10", "2024-03-10 10:00"),
])


In [9]:
cur.executemany("INSERT INTO employees VALUES (?,?,?,?,?)", [
    (101, "Sarah Connor", "Engineering", 95000, "2021-06-01"),
    (102, "John Doe",     "Sales",       72000, "2022-01-15"),
    (103, "Jane Smith",   "HR",          65000, "2020-09-10"),
    (104, "Mike Ross",    "Engineering", 88000, "2023-03-20"),
    (105, "Rachel Zane",  "Legal",       98000, "2019-11-05"),
])

In [10]:
conn.commit()

In [12]:
print("\n--- Read full table: transactions ---")
df_txn = pd.read_sql("SELECT * FROM transactions", conn)
df_txn



--- Read full table: transactions ---


,txn_id,customer_id,product,amount,status,created_at,updated_at
0,1,1,Laptop,999.99,completed,2024-01-10,2024-01-10 09:00
1,2,2,Keyboard,79.99,completed,2024-01-15,2024-01-15 14:00
2,3,3,Monitor,349.99,shipped,2024-02-01,2024-02-02 08:00
3,4,1,Mouse,25.00,completed,2024-02-10,2024-02-10 16:00
4,5,4,Tablet,599.99,completed,2024-02-20,2024-02-20 11:00
5,6,5,Headphones,199.99,completed,2024-03-01,2024-03-01 09:00
6,7,2,USB Hub,49.99,pending,2024-03-05,2024-03-05 13:00
7,8,6,Workstation,1299.99,completed,2024-03-10,2024-03-10 10:00


In [13]:
print("\n--- Filter: transactions where amount > 200 ---")
df_big = pd.read_sql(
    "SELECT * FROM transactions WHERE amount > 200 ORDER BY amount DESC",
    conn
)
df_big


--- Filter: transactions where amount > 200 ---


,txn_id,customer_id,product,amount,status,created_at,updated_at
0,8,6,Workstation,1299.99,completed,2024-03-10,2024-03-10 10:00
1,1,1,Laptop,999.99,completed,2024-01-10,2024-01-10 09:00
2,5,4,Tablet,599.99,completed,2024-02-20,2024-02-20 11:00
3,3,3,Monitor,349.99,shipped,2024-02-01,2024-02-02 08:00


In [27]:
print("\n--- Chunked read (chunk_size=3) ---")
offset = 0
chunk_size = 3
chunk_num  = 0


--- Chunked read (chunk_size=3) ---


In [28]:
final_df = pd.DataFrame() 
while True:
    chunk = pd.read_sql(
        f"SELECT * FROM transactions LIMIT {chunk_size} OFFSET {offset}",
        conn
    )
    if chunk.empty:
        break
    chunk_num += 1
    final_df = pd.concat([final_df,chunk], ignore_index = True) 
    print(f"  Chunk {chunk_num}: {len(chunk)} rows")
    offset += chunk_size

print(f"  Total chunks: {chunk_num}")

  Chunk 1: 3 rows
  Chunk 2: 3 rows
  Chunk 3: 2 rows
  Total chunks: 3


In [29]:
offset

9

In [30]:
final_df

,txn_id,customer_id,product,amount,status,created_at,updated_at
0,1,1,Laptop,999.99,completed,2024-01-10,2024-01-10 09:00
1,2,2,Keyboard,79.99,completed,2024-01-15,2024-01-15 14:00
2,3,3,Monitor,349.99,shipped,2024-02-01,2024-02-02 08:00
3,4,1,Mouse,25.00,completed,2024-02-10,2024-02-10 16:00
4,5,4,Tablet,599.99,completed,2024-02-20,2024-02-20 11:00
5,6,5,Headphones,199.99,completed,2024-03-01,2024-03-01 09:00
6,7,2,USB Hub,49.99,pending,2024-03-05,2024-03-05 13:00
7,8,6,Workstation,1299.99,completed,2024-03-10,2024-03-10 10:00


In [31]:
print("\n" + "="*60)
print(" INCREMENTAL vs FULL LOAD")
print("="*60)



 INCREMENTAL vs FULL LOAD


In [32]:
WATERMARK_FILE = Path("data/watermark.json")

In [43]:
data_dict = {
    "transaction":"2025-12-01T12:00:00Z"
}
data_dict["transaction"]

'2025-12-01T12:00:00Z'

In [47]:
def load_watermark(table_name):
    """Read the saved watermark for a table. Returns None if first run."""
    if not WATERMARK_FILE.exists():
        return None
    data = json.loads(WATERMARK_FILE.read_text())
    return data.get(table_name)

In [48]:
def save_watermark(table_name, value):
    """Save the watermark after a successful load."""
    data = {}
    if WATERMARK_FILE.exists():
        data = json.loads(WATERMARK_FILE.read_text())
    data[table_name] = str(value)
    WATERMARK_FILE.write_text(json.dumps(data, indent=2))
    print(f"  Watermark saved: {table_name} = {value}")

In [49]:
def full_load(connection, table_name):
    """Read every row from the table."""
    print(f"\n[FULL LOAD] Reading all rows from '{table_name}' ...")
    df = pd.read_sql(f"SELECT * FROM {table_name}", connection)
    print(f"  Loaded: {len(df)} rows")
    #print(df.to_string(index=False))
    # Save the latest timestamp as the new watermark
    if "updated_at" in df.columns:
        save_watermark(table_name, df["updated_at"].max())
    return df

In [50]:
def incremental_load(connection, table_name, timestamp_col="updated_at"):
    """Read only rows newer than the last watermark."""
    last_ts = load_watermark(table_name)

    if last_ts is None:
        print(f"\n[INCREMENTAL] No watermark found → running full load instead.")
        return full_load(connection, table_name)

    print(f"\n[INCREMENTAL] Loading '{table_name}' where {timestamp_col} > '{last_ts}' ...")
    df = pd.read_sql(
        f"SELECT * FROM {table_name} WHERE {timestamp_col} > '{last_ts}'",
        connection
    )

    if df.empty:
        print("  No new rows — already up to date.")
        return df

    print(f"  Loaded: {len(df)} new rows")
    #print(df.to_string(index=False))
    save_watermark(table_name, df[timestamp_col].max())
    return df

In [51]:
if WATERMARK_FILE.exists():
    WATERMARK_FILE.unlink()

In [52]:
incremental_load(conn, "transactions")


[INCREMENTAL] No watermark found → running full load instead.

[FULL LOAD] Reading all rows from 'transactions' ...
  Loaded: 8 rows
  Watermark saved: transactions = 2024-03-10 10:00


,txn_id,customer_id,product,amount,status,created_at,updated_at
0,1,1,Laptop,999.99,completed,2024-01-10,2024-01-10 09:00
1,2,2,Keyboard,79.99,completed,2024-01-15,2024-01-15 14:00
2,3,3,Monitor,349.99,shipped,2024-02-01,2024-02-02 08:00
3,4,1,Mouse,25.00,completed,2024-02-10,2024-02-10 16:00
4,5,4,Tablet,599.99,completed,2024-02-20,2024-02-20 11:00
5,6,5,Headphones,199.99,completed,2024-03-01,2024-03-01 09:00
6,7,2,USB Hub,49.99,pending,2024-03-05,2024-03-05 13:00
7,8,6,Workstation,1299.99,completed,2024-03-10,2024-03-10 10:00


In [54]:
incremental_load(conn, "transactions")


[INCREMENTAL] Loading 'transactions' where updated_at > '2024-03-10 10:00' ...
  No new rows — already up to date.


,txn_id,customer_id,product,amount,status,created_at,updated_at


In [55]:
conn.executemany("INSERT INTO transactions VALUES (?,?,?,?,?,?,?)", [
    (9,  7, "Printer",   459.99, "completed", "2024-04-01", "2024-04-01 08:00"),
    (10, 8, "Speaker",   129.99, "pending",   "2024-04-02", "2024-04-02 10:00"),
    (11, 3, "MacBook",  2499.99, "completed", "2024-04-05", "2024-04-05 14:00"),
])


In [56]:
conn.commit()

In [57]:
print("  3 rows inserted.")
incremental_load(conn, "transactions")

  3 rows inserted.

[INCREMENTAL] Loading 'transactions' where updated_at > '2024-03-10 10:00' ...
  Loaded: 3 new rows
  Watermark saved: transactions = 2024-04-05 14:00


,txn_id,customer_id,product,amount,status,created_at,updated_at
0,9,7,Printer,459.99,completed,2024-04-01,2024-04-01 08:00
1,10,8,Speaker,129.99,pending,2024-04-02,2024-04-02 10:00
2,11,3,MacBook,2499.99,completed,2024-04-05,2024-04-05 14:00


In [58]:
print("─" * 40)
print("RUN 4 — Force full load (reloads everything)")
full_load(conn, "transactions")

────────────────────────────────────────
RUN 4 — Force full load (reloads everything)

[FULL LOAD] Reading all rows from 'transactions' ...
  Loaded: 11 rows
  Watermark saved: transactions = 2024-04-05 14:00


,txn_id,customer_id,product,amount,status,created_at,updated_at
0,1,1,Laptop,999.99,completed,2024-01-10,2024-01-10 09:00
1,2,2,Keyboard,79.99,completed,2024-01-15,2024-01-15 14:00
2,3,3,Monitor,349.99,shipped,2024-02-01,2024-02-02 08:00
3,4,1,Mouse,25.00,completed,2024-02-10,2024-02-10 16:00
4,5,4,Tablet,599.99,completed,2024-02-20,2024-02-20 11:00
5,6,5,Headphones,199.99,completed,2024-03-01,2024-03-01 09:00
6,7,2,USB Hub,49.99,pending,2024-03-05,2024-03-05 13:00
7,8,6,Workstation,1299.99,completed,2024-03-10,2024-03-10 10:00
8,9,7,Printer,459.99,completed,2024-04-01,2024-04-01 08:00
9,10,8,Speaker,129.99,pending,2024-04-02,2024-04-02 10:00


In [59]:
print("\n" + "="*60)
print("  SECTION 5 — CAPSTONE: MULTI-SOURCE PIPELINE")
print("="*60)


  SECTION 5 — CAPSTONE: MULTI-SOURCE PIPELINE


In [60]:
print("\nStep 1: Read files")
df_c = pd.read_csv(RAW / "customers.csv")


Step 1: Read files


In [62]:
df_c

,customer_id,name,email,city,tier
0,1,Alice Johnson,alice@email.com,New York,Gold
1,2,Bob Smith,bob@email.com,Chicago,Silver
2,3,Carol White,carol@email.com,Los Angeles,Gold
3,4,David Lee,david@email.com,Houston,Bronze
4,5,Eva Brown,eva@email.com,Phoenix,Silver


In [63]:
df_c["source"] = "file:customers"

In [64]:
df_c

,customer_id,name,email,city,tier,source
0,1,Alice Johnson,alice@email.com,New York,Gold,file:customers
1,2,Bob Smith,bob@email.com,Chicago,Silver,file:customers
2,3,Carol White,carol@email.com,Los Angeles,Gold,file:customers
3,4,David Lee,david@email.com,Houston,Bronze,file:customers
4,5,Eva Brown,eva@email.com,Phoenix,Silver,file:customers


In [65]:
with open(RAW / "orders.json") as f:
    df_o = pd.DataFrame(json.load(f))

In [66]:
df_o["source"] = "file:orders"

In [67]:
df_o

,order_id,customer_id,product,qty,price,status,source
0,101,1,Laptop,1,999.99,delivered,file:orders
1,102,2,Mouse,2,25.00,shipped,file:orders
2,103,3,Monitor,1,349.99,pending,file:orders
3,104,1,Keyboard,1,79.99,delivered,file:orders
4,105,5,Webcam,1,89.99,shipped,file:orders


In [68]:
file_df = pd.concat([df_c, df_o], ignore_index=True)
print(f"  Files → {len(file_df)} rows")

  Files → 10 rows


In [69]:
file_df

,customer_id,name,email,city,tier,source,order_id,product,qty,price,status
0,1,Alice Johnson,alice@email.com,New York,Gold,file:customers,NaN,NaN,NaN,NaN,NaN
1,2,Bob Smith,bob@email.com,Chicago,Silver,file:customers,NaN,NaN,NaN,NaN,NaN
2,3,Carol White,carol@email.com,Los Angeles,Gold,file:customers,NaN,NaN,NaN,NaN,NaN
3,4,David Lee,david@email.com,Houston,Bronze,file:customers,NaN,NaN,NaN,NaN,NaN
4,5,Eva Brown,eva@email.com,Phoenix,Silver,file:customers,NaN,NaN,NaN,NaN,NaN
5,1,NaN,NaN,NaN,NaN,file:orders,101.0,Laptop,1.0,999.99,delivered
6,2,NaN,NaN,NaN,NaN,file:orders,102.0,Mouse,2.0,25.00,shipped
7,3,NaN,NaN,NaN,NaN,file:orders,103.0,Monitor,1.0,349.99,pending
8,1,NaN,NaN,NaN,NaN,file:orders,104.0,Keyboard,1.0,79.99,delivered
9,5,NaN,NaN,NaN,NaN,file:orders,105.0,Webcam,1.0,89.99,shipped


In [70]:
print("\nStep 2: Fetch from API")


Step 2: Fetch from API


In [72]:
BASE_URL = "https://jsonplaceholder.typicode.com"
def api_get(endpoint, params=None):
    url = BASE_URL + endpoint
    for attempt in range(3):
        try:
            r = requests.get(url, params=params, timeout=10)
            r.raise_for_status()
            return r.json()
        except requests.RequestException as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            if attempt == 2:
                raise
    return []

In [73]:
try:
    raw_users = api_get("/users")
    api_df = pd.json_normalize(raw_users)[["id", "name", "email"]]
    api_df.columns = ["customer_id", "name", "email"]
    api_df["source"] = "api:users"
    print(f"  API → {len(api_df)} rows")
except Exception as e:
    print(f"  API failed ({e}), skipping.")
    api_df = pd.DataFrame()


  API → 10 rows


In [74]:
pd.json_normalize(raw_users)[["id", "name", "email"]]

,id,name,email
0,1,Leanne Graham,Sincere@april.biz
1,2,Ervin Howell,Shanna@melissa.tv
2,3,Clementine Bauch,Nathan@yesenia.net
3,4,Patricia Lebsack,Julianne.OConner@kory.org
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca
5,6,Mrs. Dennis Schulist,Karley_Dach@jasper.info
6,7,Kurtis Weissnat,Telly.Hoeger@billy.biz
7,8,Nicholas Runolfsdottir V,Sherwood@rosamond.me
8,9,Glenna Reichert,Chaim_McDermott@dana.io
9,10,Clementina DuBuque,Rey.Padberg@karina.biz


In [75]:
# Step 3 — Database (incremental)
print("\nStep 3: Database (incremental)")
db_df = pd.read_sql("SELECT * FROM transactions", conn)
db_df["source"] = "db:transactions"
print(f"  Database → {len(db_df)} rows")


Step 3: Database (incremental)
  Database → 11 rows


In [76]:
db_df

,txn_id,customer_id,product,amount,status,created_at,updated_at,source
0,1,1,Laptop,999.99,completed,2024-01-10,2024-01-10 09:00,db:transactions
1,2,2,Keyboard,79.99,completed,2024-01-15,2024-01-15 14:00,db:transactions
2,3,3,Monitor,349.99,shipped,2024-02-01,2024-02-02 08:00,db:transactions
3,4,1,Mouse,25.00,completed,2024-02-10,2024-02-10 16:00,db:transactions
4,5,4,Tablet,599.99,completed,2024-02-20,2024-02-20 11:00,db:transactions
5,6,5,Headphones,199.99,completed,2024-03-01,2024-03-01 09:00,db:transactions
6,7,2,USB Hub,49.99,pending,2024-03-05,2024-03-05 13:00,db:transactions
7,8,6,Workstation,1299.99,completed,2024-03-10,2024-03-10 10:00,db:transactions
8,9,7,Printer,459.99,completed,2024-04-01,2024-04-01 08:00,db:transactions
9,10,8,Speaker,129.99,pending,2024-04-02,2024-04-02 10:00,db:transactions


In [77]:
# Step 4 — Merge
from datetime import datetime
print("\nStep 4: Merge all sources")
all_dfs = [df for df in [file_df, api_df, db_df] if not df.empty]
merged = pd.concat(all_dfs, ignore_index=True)
merged.drop_duplicates(inplace=True)
merged["ingested_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"  Merged → {len(merged)} rows, {len(merged.columns)} columns")


Step 4: Merge all sources
  Merged → 31 rows, 16 columns


In [78]:
merged

,customer_id,name,email,city,tier,source,order_id,product,qty,price,status,txn_id,amount,created_at,updated_at,ingested_at
0,1,Alice Johnson,alice@email.com,New York,Gold,file:customers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-02 09:45:32
1,2,Bob Smith,bob@email.com,Chicago,Silver,file:customers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-02 09:45:32
2,3,Carol White,carol@email.com,Los Angeles,Gold,file:customers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-02 09:45:32
3,4,David Lee,david@email.com,Houston,Bronze,file:customers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-02 09:45:32
4,5,Eva Brown,eva@email.com,Phoenix,Silver,file:customers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-02 09:45:32
5,1,NaN,NaN,NaN,NaN,file:orders,101.0,Laptop,1.0,999.99,delivered,NaN,NaN,NaN,NaN,2026-03-02 09:45:32
6,2,NaN,NaN,NaN,NaN,file:orders,102.0,Mouse,2.0,25.00,shipped,NaN,NaN,NaN,NaN,2026-03-02 09:45:32
7,3,NaN,NaN,NaN,NaN,file:orders,103.0,Monitor,1.0,349.99,pending,NaN,NaN,NaN,NaN,2026-03-02 09:45:32
8,1,NaN,NaN,NaN,NaN,file:orders,104.0,Keyboard,1.0,79.99,delivered,NaN,NaN,NaN,NaN,2026-03-02 09:45:32
9,5,NaN,NaN,NaN,NaN,file:orders,105.0,Webcam,1.0,89.99,shipped,NaN,NaN,NaN,NaN,2026-03-02 09:45:32


In [80]:
print("\nStep 5: Save output")
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out = PROCESSED / f"pipeline_output_{ts}.csv"
merged.to_csv(out, index=False)
print(f"  Saved → {out}")


Step 5: Save output
  Saved → data\processed\pipeline_output_20260302_094701.csv


In [81]:
source_counts = merged["source"].value_counts().to_dict()
for src, cnt in source_counts.items():
    print(f"    {src}: {cnt} rows")


    db:transactions: 11 rows
    api:users: 10 rows
    file:orders: 5 rows
    file:customers: 5 rows


In [82]:
conn.close()